# Commodity Intelligence Pipeline — Brazilian Soy, End to End

**For a quant hedge fund's alternative-data desk.**

Brazil grows ~40% of the world's soybeans. CONAB's monthly crop report
and USDA WASDE hit markets with a week+ lag; a fund that can estimate
Mato Grosso pod-fill conditions in near-real-time from Sentinel-2 has a
multi-week informational edge on the soy complex, oilseed crush margins,
and the BRL cross.

This notebook is the **production-shape pipeline**, compressed into one
file:

1. **Define growing regions** — bounding polygons for Brazil's top five
   soy-producing states (MT, PR, RS, GO, MS).
2. **Ingest Sentinel-2** — STAC query for L2A scenes over each region,
   growing season only (Oct–Apr).
3. **RasterFlow crop health** — per-scene NDVI via `RS_MapAlgebra` over
   the Red and NIR bands; production pattern shown inline.
4. **Zonal statistics** — `RS_ZonalStats` aggregates NDVI from pixel to
   region × week.
5. **Signal construction** — acreage-weighted NDVI anomaly vs. 5-year
   climatology; pod-fill-window (Jan–Feb) anomaly drives the final signal.
6. **Ship to Delta Lake** — a weekly signal table the quant stack reads
   directly: one row per (week, region) with anomaly, z-score, acreage.

**Target.** `wb_delta.commodity_intel_demo.br_soy_weekly_signals` — Delta
format so Databricks / Spark SQL / Trino / DuckDB workflows read without
conversion.

> **Demo scope.** The STAC scene query is real. The per-scene NDVI and
> zonal-stats cells show the SQL pattern a production job runs; the
> weekly region × week signal series below is synthesized with a
> realistic mid-January flash-drought anomaly in MT/GO so the downstream
> signal + Delta write show what the quant system actually consumes.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)
import numpy as np
import pandas as pd

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

# Delta target — parameterized so the customer can redirect
DELTA_CATALOG  = "wb_delta"
DELTA_DATABASE = "commodity_intel_demo"
DELTA_TABLE    = "br_soy_weekly_signals"
DELTA_FQN      = f"{DELTA_CATALOG}.{DELTA_DATABASE}.{DELTA_TABLE}"

## 2. Define the Growing Regions

Five states cover ~80% of Brazilian soybean production. Each gets an
approximate bounding envelope + a 2024/25-season planted-area figure
(million hectares) from CONAB public releases. Production replaces
these envelopes with actual IBGE state polygons; the API surface is
identical.

In [ ]:
# (code, name,                min_lon, min_lat, max_lon, max_lat, planted_Mha)
regions = [
    ("MT", "Mato Grosso",        -61.5, -17.5, -50.3,  -7.3, 12.60),
    ("PR", "Paraná",              -54.6, -26.7, -48.0, -22.5,  5.80),
    ("RS", "Rio Grande do Sul",   -57.6, -33.7, -49.7, -27.1,  6.70),
    ("GO", "Goiás",               -53.2, -19.5, -45.9, -12.4,  4.75),
    ("MS", "Mato Grosso do Sul",  -58.2, -24.1, -50.9, -17.2,  4.00),
]

region_schema = StructType([
    StructField("region_code",  StringType()),
    StructField("region_name",  StringType()),
    StructField("min_lon",      DoubleType()),
    StructField("min_lat",      DoubleType()),
    StructField("max_lon",      DoubleType()),
    StructField("max_lat",      DoubleType()),
    StructField("planted_Mha",  DoubleType()),
])

regions_df = sedona.createDataFrame(regions, region_schema) \
    .withColumn(
        "geometry",
        f.expr("ST_MakeEnvelope(min_lon, min_lat, max_lon, max_lat, 4326)")
    )
regions_df.createOrReplaceTempView("soy_regions")

print(f"Regions: {regions_df.count()}, "
      f"total planted area: "
      f"{regions_df.agg(f.sum('planted_Mha')).first()[0]:.1f} Mha")
regions_df.select("region_code", "region_name", "planted_Mha").show(truncate=False)

## 3. Stage 1 — Ingest Sentinel-2 via STAC

Hit the public Element 84 AWS STAC endpoint for L2A scenes covering
Mato Grosso during the 2024-25 growing season (Oct 2024 - Apr 2025).
The same call pattern runs per-region in production.

In [ ]:
from sedona.stac.client import Client

stac = Client.open("https://earth-search.aws.element84.com/v1")
mt_bbox = [-61.5, -17.5, -50.3, -7.3]

# The Sedona STAC client wraps endpoint errors in opaque RuntimeErrors;
# retry across datetime formats before giving up.
items_list = []
for dt_expr in [
    "2024-10-01T00:00:00Z/2025-04-30T00:00:00Z",
    "2024-10-01/2025-04-30",
    "2025",
]:
    try:
        items = stac.search(
            collection_id="sentinel-2-c1-l2a",
            bbox=mt_bbox,
            datetime=dt_expr,
            max_items=500,
            return_dataframe=False,
        )
        items_list = list(items)
        print(f"STAC search succeeded: datetime={dt_expr!r}")
        break
    except Exception as e:
        print(f"  retry: {dt_expr!r} -> {type(e).__name__}")

if items_list:
    print(f"\nSentinel-2 scenes over Mato Grosso, 2024-25 season: {len(items_list)}")
    for item in sorted(items_list, key=lambda x: x.datetime, reverse=True)[:5]:
        print(f"  {item.datetime.strftime('%Y-%m-%d')}  {item.id}")
else:
    print("STAC endpoint unavailable — continuing with demo narrative.")
    print("Sentinel-2 nominally delivers 300+ scenes per season per BR state.")

## 4. Stage 2 — RasterFlow Crop Health: NDVI via `RS_MapAlgebra`

Per-scene NDVI is a single map-algebra call over Red (B04) and NIR (B08)
bands. The snippet below is the production SQL pattern; the demo skips
the pixel loop to keep runtime short.

```python
# Load scenes as an out-db raster dataframe (STAC → rast column)
scenes_df = sedona.read.format("stac").load(
    "https://earth-search.aws.element84.com/v1/collections/sentinel-2-c1-l2a"
)

# Compute NDVI per scene — one SQL call, no Python loop
ndvi_df = scenes_df.selectExpr(
    "id", "datetime",
    "RS_MapAlgebra(rast, 'FLOAT32', "
    "  'out = (rast[7] - rast[3]) / (rast[7] + rast[3]);') AS ndvi"
)
```

(Band indices: `rast[3]` = B04 Red, `rast[7]` = B08 NIR in the L2A asset
order.) The output `ndvi` column is itself a raster, ready for zonal
aggregation in the next cell.

## 5. Stage 3 — Zonal Statistics per Region via `RS_ZonalStats`

`RS_ZonalStats(raster, zone_geom, stat)` aggregates pixels under each
polygon to a single number. Joined across the scene table and the region
table, it yields one NDVI observation per (region × scene_date). A
downstream `GROUP BY week` collapses to weekly cadence.

```python
weekly_ndvi = sedona.sql("""
    SELECT
        r.region_code,
        r.region_name,
        DATE_TRUNC('WEEK', s.datetime) AS week,
        AVG(RS_ZonalStats(s.ndvi, r.geometry, 'mean')) AS region_ndvi,
        COUNT(*)                                        AS scene_count
    FROM ndvi_scenes s
    JOIN soy_regions r
      ON ST_Intersects(RS_Envelope(s.ndvi), r.geometry)
    GROUP BY r.region_code, r.region_name, DATE_TRUNC('WEEK', s.datetime)
""")
```

The demo stops short of running this on the full ~1500-scene season
stack; Section 6 synthesizes the same-shape output.

## 6. Stage 4 — Weekly Signal Construction

Build 26 weeks of region × week NDVI for the current season and a
5-year climatology. MT and GO are injected with a flash-drought dip
centered on the Jan-Feb pod-fill window — the yield-determining stage
the fund is listening for.

In [ ]:
rng = np.random.default_rng(11)
WEEKS = 26

# Brazil soy season: plant Oct, pod-fill Jan-Feb, harvest Mar-Apr
weeks = pd.date_range("2024-10-28", periods=WEEKS, freq="W-MON")
week_idx = np.arange(WEEKS)

# Reference NDVI curve: rise through Nov, peak week 12-15 (mid-Jan to early Feb),
# decline through senescence + harvest
ref_ndvi = 0.25 + 0.55 * np.exp(-((week_idx - 13) / 6.0) ** 2)

STRESSED_REGIONS = {"MT", "GO"}
rows = []
for code, name, _, _, _, _, planted in regions:
    bias = rng.normal(0, 0.012)
    climo   = np.clip(ref_ndvi + bias, 0, 0.95)
    noise   = rng.normal(0, 0.010, WEEKS)
    current = climo + noise
    if code in STRESSED_REGIONS:
        # Dip centered on week 14 (late Jan), 5-week duration
        stress = -0.14 * np.exp(-((week_idx - 14) / 3.0) ** 2)
        current = current + stress
    current = np.clip(current, 0, 0.95)

    for w_idx, wk_date, climo_v, cur_v in zip(week_idx, weeks, climo, current):
        rows.append((
            wk_date.strftime("%Y-%m-%d"),
            int(w_idx + 1),
            code, name,
            float(planted),
            float(round(climo_v, 4)),
            float(round(cur_v, 4)),
            float(round(cur_v - climo_v, 4)),
        ))

signal_schema = StructType([
    StructField("week_start",     StringType()),
    StructField("season_week",    IntegerType()),
    StructField("region_code",    StringType()),
    StructField("region_name",    StringType()),
    StructField("planted_Mha",    DoubleType()),
    StructField("ndvi_climo",     DoubleType()),
    StructField("ndvi_current",   DoubleType()),
    StructField("ndvi_anomaly",   DoubleType()),
])

weekly_df = sedona.createDataFrame(rows, signal_schema)
weekly_df.createOrReplaceTempView("weekly_region_ndvi")

# Anomaly z-score within the region's own 26-week window, plus a
# national-level acreage-weighted anomaly for that week.
signal_df = sedona.sql("""
    WITH region_stats AS (
        SELECT
            region_code,
            AVG(ndvi_anomaly) AS anom_mean,
            STDDEV(ndvi_anomaly) AS anom_std
        FROM weekly_region_ndvi
        GROUP BY region_code
    ),
    national AS (
        SELECT
            week_start,
            ROUND(SUM(ndvi_anomaly * planted_Mha) / SUM(planted_Mha), 4)
                AS national_weighted_anomaly
        FROM weekly_region_ndvi
        GROUP BY week_start
    )
    SELECT
        w.week_start,
        w.season_week,
        w.region_code,
        w.region_name,
        w.planted_Mha,
        w.ndvi_climo,
        w.ndvi_current,
        w.ndvi_anomaly,
        ROUND((w.ndvi_anomaly - rs.anom_mean) / NULLIF(rs.anom_std, 0), 3)
            AS region_anomaly_zscore,
        n.national_weighted_anomaly,
        CASE
            WHEN w.ndvi_anomaly <= -0.08 THEN 'SEVERE_STRESS'
            WHEN w.ndvi_anomaly <= -0.04 THEN 'ELEVATED_STRESS'
            WHEN w.ndvi_anomaly <= -0.02 THEN 'MILD_STRESS'
            WHEN w.ndvi_anomaly <   0.02 THEN 'NORMAL'
            ELSE                              'ABOVE_TREND'
        END AS signal_tier,
        CURRENT_TIMESTAMP() AS generated_at
    FROM weekly_region_ndvi w
    JOIN region_stats rs USING (region_code)
    JOIN national     n  USING (week_start)
""").cache()
signal_df.createOrReplaceTempView("signal_rows")

print(f"Signal rows: {signal_df.count()} "
      f"({len(regions)} regions × {WEEKS} weeks)")

signal_df.filter("season_week BETWEEN 12 AND 17") \
         .orderBy("season_week", "region_code") \
         .select("week_start", "region_code", "ndvi_current",
                 "ndvi_anomaly", "region_anomaly_zscore", "signal_tier") \
         .show(30, truncate=False)

## 7. Season-to-Date Rollup — What the PM Sees on Monday Morning

Before writing the full weekly table, show the rollup the portfolio
manager scans: pod-fill-window anomaly per region, ranked by severity.

In [ ]:
sedona.sql("""
    SELECT
        region_code,
        region_name,
        planted_Mha,
        ROUND(AVG(ndvi_current),  3)      AS podfill_avg_ndvi,
        ROUND(AVG(ndvi_climo),    3)      AS podfill_climo,
        ROUND(AVG(ndvi_anomaly),  3)      AS podfill_anomaly,
        ROUND(AVG(region_anomaly_zscore), 2) AS podfill_zscore,
        MAX(signal_tier)                  AS peak_signal_tier
    FROM signal_rows
    WHERE season_week BETWEEN 12 AND 18
    GROUP BY region_code, region_name, planted_Mha
    ORDER BY podfill_anomaly ASC
""").show(truncate=False)

## 8. Stage 5 — Write to Delta Lake

`CREATE DATABASE IF NOT EXISTS` + `CREATE OR REPLACE TABLE … USING delta`
lands the signal in a Delta-format table in the `wb_delta` catalog.
Databricks, Spark SQL, Trino, DuckDB, and Polars can all read it
directly — this is the quant system's ingestion boundary.

In [ ]:
sedona.sql(f"CREATE DATABASE IF NOT EXISTS {DELTA_CATALOG}.{DELTA_DATABASE}")

sedona.sql(f"""
    CREATE OR REPLACE TABLE {DELTA_FQN}
    USING delta
    AS SELECT
        week_start,
        season_week,
        region_code,
        region_name,
        planted_Mha,
        ndvi_climo,
        ndvi_current,
        ndvi_anomaly,
        region_anomaly_zscore,
        national_weighted_anomaly,
        signal_tier,
        generated_at
    FROM signal_rows
""")

print(f"Wrote Delta table: {DELTA_FQN}")
sedona.sql(f"DESCRIBE TABLE {DELTA_FQN}").show(truncate=False)

## 9. Validation — Read Back from Delta

Confirm the Delta round-trip and show the rows a downstream quant
backtester would load.

In [ ]:
summary = sedona.sql(f"""
    SELECT
        COUNT(*)                          AS row_count,
        COUNT(DISTINCT region_code)       AS regions,
        COUNT(DISTINCT week_start)        AS weeks,
        MIN(week_start)                   AS first_week,
        MAX(week_start)                   AS last_week,
        ROUND(MIN(ndvi_anomaly), 3)       AS worst_anomaly,
        ROUND(MAX(ndvi_anomaly), 3)       AS best_anomaly
    FROM {DELTA_FQN}
""")
summary.show(truncate=False)

sedona.sql(f"""
    SELECT week_start, region_code, ndvi_current, ndvi_anomaly,
           region_anomaly_zscore, signal_tier
    FROM {DELTA_FQN}
    WHERE signal_tier IN ('SEVERE_STRESS', 'ELEVATED_STRESS')
    ORDER BY ndvi_anomaly ASC
    LIMIT 15
""").show(truncate=False)

## 10. Preview on a Map

In [ ]:
latest_week = sedona.sql(
    f"SELECT MAX(week_start) AS wk FROM {DELTA_FQN}"
).first().wk

map_df = sedona.sql(f"""
    SELECT
        s.region_code,
        s.region_name,
        s.planted_Mha,
        s.ndvi_current,
        s.ndvi_anomaly,
        s.signal_tier,
        r.geometry
    FROM {DELTA_FQN} s
    JOIN soy_regions r USING (region_code)
    WHERE s.week_start = '{latest_week}'
""")
SedonaKepler.create_map(map_df, name="BR Soy — Latest Weekly Signal")

---

## From demo to production

Same Wherobots surface, swap in live feeds:

| Stage | Demo | Production |
|---|---|---|
| Regions | Bounding envelopes for 5 states | IBGE state / município polygons + CONAB soybean mask |
| Imagery | STAC search shown; inference deferred | `RS_MapAlgebra` over S2 L2A stack via `build_and_predict_mosaic_recipe` |
| Zonal stats | SQL pattern shown; series synthesized | `RS_ZonalStats` over NDVI rasters × region polygons |
| Climatology | Smooth Gaussian reference | 5-year MODIS/HLS rolling baseline |
| Signal | Weekly anomaly + z-score + tier | Same, plus forward yield regression model |
| Sink | `wb_delta.commodity_intel_demo.*` | Fund's own Databricks UC / S3 Delta location |

## Downstream consumption

The Delta table at `wb_delta.commodity_intel_demo.br_soy_weekly_signals`
is the integration boundary. A quant backtester reads it with:

```python
# Spark / Databricks
spark.read.table("wb_delta.commodity_intel_demo.br_soy_weekly_signals")

# Or direct Delta from Python / Polars
from deltalake import DeltaTable
DeltaTable("s3://.../br_soy_weekly_signals").to_pandas()
```